In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd, xgboost as xgb, shap

SEED = 42
np.random.seed(SEED)

ROOT      = Path(r"C:\Fintech-Project\graph_AML_pipeline")
DATA_PROC = ROOT / "data" / "processed"
OUT_DIR   = ROOT / "outputs"
(OUT_DIR / "shap").mkdir(parents=True, exist_ok=True)

def find_one(name):
    hits = list(OUT_DIR.rglob(name))
    assert len(hits) == 1, f"{name}: expected 1 match, found {len(hits)} -> {hits}"
    return hits[0]

A = {n: find_one(n) for n in [
    "m1_baseline.json", "m2_graph_xgb.json",
    "m1_test_probs.parquet", "m2_test_probs.parquet",
    "06_m1_features.json", "07_m2_features.json",
]}
for k, v in A.items():
    print(f"{k:26s} {v.relative_to(ROOT)}")
print(f"\nxgboost {xgb.__version__} | shap {shap.__version__} | pandas {pd.__version__}")

c:\Users\chara\anaconda3\envs\graphaml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


m1_baseline.json           outputs\models\m1_baseline.json
m2_graph_xgb.json          outputs\models\m2_graph_xgb.json
m1_test_probs.parquet      outputs\models\m1_test_probs.parquet
m2_test_probs.parquet      outputs\models\m2_test_probs.parquet
06_m1_features.json        outputs\models\06_m1_features.json
07_m2_features.json        outputs\models\07_m2_features.json

xgboost 2.1.4 | shap 0.46.0 | pandas 2.2.3


In [3]:
m1 = xgb.Booster(); m1.load_model(str(A["m1_baseline.json"]))
m2 = xgb.Booster(); m2.load_model(str(A["m2_graph_xgb.json"]))

TAB_FEATS = json.load(open(A["06_m1_features.json"]))
M2_FEATS  = json.load(open(A["07_m2_features.json"]))
GRAPH_FEATS = [f for f in M2_FEATS if f.startswith(("from_g_", "to_g_"))]

assert len(TAB_FEATS) == 10 and len(M2_FEATS) == 26 and len(GRAPH_FEATS) == 16
assert M2_FEATS[:10] == TAB_FEATS, "shared features must stay in M1 order"
print(f"M1 {len(TAB_FEATS)} features · M2 {len(M2_FEATS)} features · graph family {len(GRAPH_FEATS)}")
print(f"shared features for H4(a): {len(TAB_FEATS)}")

M1 10 features · M2 26 features · graph family 16
shared features for H4(a): 10


In [4]:
df = pd.read_parquet(DATA_PROC / "05_features_full.parquet")
assert len(df) == 5_078_345, "row count does not match the registered artefact"

m1p = pd.read_parquet(A["m1_test_probs.parquet"])
m2p = pd.read_parquet(A["m2_test_probs.parquet"])

# read the flag — never re-derive the wind-down mask
main_idx = m1p.index[m1p["is_main"].values]
assert m1p.index.equals(m2p.index), "M1 and M2 probability files are not on one index"
assert (m1p["is_main"].values == m2p["is_main"].values).all()
assert len(main_idx) == 760_531, f"D1 violated: got {len(main_idx):,}, expected test-main"
assert df.loc[main_idx, "Is Laundering"].sum() == 906

# D1: 10,000 rows sampled from TEST-MAIN, identical indices for both models
sample_idx = pd.Index(main_idx).to_series().sample(10_000, random_state=SEED).index
X_sample = df.loc[sample_idx]

assert sample_idx.isin(main_idx).all(), "sample escaped test-main"
np.save(OUT_DIR / "shap" / "08_test_indices.npy", sample_idx.values)
print(f"sample {len(sample_idx):,} rows · {int(X_sample['Is Laundering'].sum())} illicit "
      f"· drawn from test-main {len(main_idx):,}")

sample 10,000 rows · 8 illicit · drawn from test-main 760,531


In [5]:
train = df[df["split"] == "train"]
n_bg  = 1000
n_pos = max(1, int(n_bg * train["Is Laundering"].mean()))
bg_pos = train[train["Is Laundering"] == 1].sample(n_pos, random_state=SEED)
bg_neg = train[train["Is Laundering"] == 0].sample(n_bg - n_pos, random_state=SEED)
background = pd.concat([bg_pos, bg_neg]).sample(frac=1, random_state=SEED)

assert len(background) == 1000
print(f"background {len(background):,} rows · {int(background['Is Laundering'].sum())} illicit")

background 1,000 rows · 1 illicit


In [7]:
# derive the three time features and encode the categoricals, exactly as notebook 07 did
df["hour"]        = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

CAT_COLS = ["Receiving Currency", "Payment Currency", "Payment Format"]
for c in CAT_COLS:
    df[c] = df[c].astype("category").cat.codes

# the codes must be identical to notebook 06's, or every SHAP value is misattributed
maps = json.load(open(find_one("06_category_maps.json")))
for c in CAT_COLS:
    expected = {int(k): v for k, v in maps[c].items()}
    actual = dict(enumerate(pd.read_parquet(DATA_PROC / "05_features_full.parquet",
                                            columns=[c])[c].astype("category").cat.categories))
    assert expected == actual, f"category codes differ from notebook 06 for {c}"
    print(f"{c} -> {len(actual)} categories, codes match")

# rebuild both frames on the completed table; same indices, same seed, same rows
X_sample = df.loc[sample_idx]

train  = df[df["split"] == "train"]
n_bg   = 1000
n_pos  = max(1, int(n_bg * train["Is Laundering"].mean()))
bg_pos = train[train["Is Laundering"] == 1].sample(n_pos, random_state=SEED)
bg_neg = train[train["Is Laundering"] == 0].sample(n_bg - n_pos, random_state=SEED)
background = pd.concat([bg_pos, bg_neg]).sample(frac=1, random_state=SEED)

missing = [f for f in M2_FEATS if f not in df.columns]
assert not missing, f"still missing: {missing}"
assert X_sample[M2_FEATS].isna().sum().sum() == 0
assert background[M2_FEATS].isna().sum().sum() == 0
print(f"\nsample {len(X_sample):,} rows · {int(X_sample['Is Laundering'].sum())} illicit")
print(f"background {len(background):,} rows · {int(background['Is Laundering'].sum())} illicit")

Receiving Currency -> 15 categories, codes match
Payment Currency -> 15 categories, codes match
Payment Format -> 7 categories, codes match

sample 10,000 rows · 8 illicit
background 1,000 rows · 1 illicit


In [8]:
import time

thr_m1 = json.load(open(find_one("06_m1_threshold.json")))
thr_m2 = json.load(open(find_one("07_m2_threshold.json")))
n1, n2 = thr_m1["n_trees_used"], thr_m2["n_trees_used"]
assert n1 == 482 and n2 == 430, f"tree counts off-spec: {n1}, {n2}"

# slice to the same trees Notebooks 06/07 used via iteration_range
m1_used, m2_used = m1[:n1], m2[:n2]

# the sliced models must reproduce the probabilities already on disk
p1 = m1_used.predict(xgb.DMatrix(X_sample[TAB_FEATS]))
p2 = m2_used.predict(xgb.DMatrix(X_sample[M2_FEATS]))
d1 = np.abs(p1 - m1p.loc[sample_idx, "m1_prob"].values).max()
d2 = np.abs(p2 - m2p.loc[sample_idx, "m2_prob"].values).max()

print(f"M1 {n1} trees · max prob diff {d1:.3e}")
print(f"M2 {n2} trees · max prob diff {d2:.3e}")
assert d1 < 1e-6 and d2 < 1e-6, "sliced model does not reproduce the scored predictions"
print("\nSHAP will explain exactly the models H1 was decided on")

M1 482 trees · max prob diff 0.000e+00
M2 430 trees · max prob diff 0.000e+00

SHAP will explain exactly the models H1 was decided on
